# Actividad 2
__Curso:__ Tópicos avanzados en Inteligencia Artificial 1

__Programa:__ MIA 2-2025

__Profesor:__ Anthony D. Cho

__Ayudante corrector:__ Luis Oliveros

## Instrucciones
* La actividad debe ser realizada por los grupos capstones
* Por favor responder en este mismo notebook (una entrega por grupo)
* Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo actividad2_Sakuragi_Mitsui_Rukagua_Sendoh.ipynb
* Subir el archivo al link de entrega Actividad 2 en webcursos que será habilitado

__Fecha de entrega:__ Fecha límite de entrega 06 de septiembre de 2026 - 23:59 horas chile.

__Integrantes:__ (RUT, Nombre y Apellido)

* 13.257.556-8, Ricardo Lopez
* 16.789.149-7, Camilo Muñoz


In [1]:
## Descomente la siguiente linea si es necesrio instalar la siguiente libreria
#python -m pip install kagglehub

## Librerias

In [ ]:
import kagglehub
from pathlib import Path
import matplotlib.pyplot as plt
from pandas import read_csv

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

## Agregar las otras librerias que necesiten


## Funciones personalizadas

In [ ]:
def plot_history(history, width=12, height=6):
  """
  DESCRIPTION:
    History performance of the keras model

  INPUT:
    @param history: history of performance of fitted model
    @type history: tensorflow.python.keras.callbacks.History

  OUTPUT:
    A graphic
  """

  ## Metrics keys stored in tensorflow object
  keys = list(history.history.keys())

  ## Number of epoch used for fit the model
  epoch = range(1, len(history.epoch) +1)

  ## Check if validation set was used.
  withValidation = False
  for key in keys:
    if 'val' in key:
      withValidation = True

  ## Number of metrics
  nMetrics = len(keys)
  if withValidation:
    nMetrics = nMetrics//2

  ## Plot-space instance
  plt.figure(figsize=(width, height))

  for i in range(nMetrics):
    plt.subplot(nMetrics, 1, i+1)

    ## Plot (train) metric value
    labelMetric = keys[i]
    metric = history.history[keys[i]]
    plt.plot(epoch, metric, 'o-', label=labelMetric)

    if withValidation:
      ## Plot (validation) metric value
      labelMetricVal = keys[i+nMetrics]
      metricVal = history.history[keys[i+nMetrics]]
      plt.plot(epoch, metricVal, 'o-', label=labelMetricVal)

    plt.xlim(epoch[0], epoch[-1])
    plt.legend()
    plt.grid()

  plt.xlabel('Epoch')
  plt.show()

## Dataset

<center>
    <img src=https://www.danielle-moss.com/wp-content/uploads/2021/04/Horizontal-Homepage-4.png width=800>
</center>

El conjunto de datos contiene alrededor de 183 mil revisiones de productos de bebes con sus respectivas valoración recopilados desde la página de Amazon,

* La data y detalles está completamente disponible en [Kaggle: Reviews of Amazon Baby Products](https://www.kaggle.com/datasets/sameersmahajan/reviews-of-amazon-baby-products)

#### Carga de datos

In [ ]:
## Descarga del dataset desde kaggle
path = kagglehub.dataset_download("sameersmahajan/reviews-of-amazon-baby-products")

## Declaración de la ruta de los datos
src = Path(path)
file = Path.joinpath(src, 'amazon_baby.csv')

## Load data
data = read_csv(file)

## Se asigna como sentimiento: Ratings {1,2} a 0 (negativo), y el resto a 1 (positivo)
data['sentiment'] = data['rating'].apply(lambda x: 0 if x in [1, 2, 3] else 1)

## Separar el predictor (review) y target (sentimient)
X, y = data['review'], data['sentiment']

## Display first 4 records
data.head(4)

#### Partición de los datos

In [ ]:
## Data partition
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)

## Convert DataFrame to list
X_train = [str(p) for p in X_train]
X_test = [str(p) for p in X_test]

## Display shape
print('(train) X: {}, y: {}'.format(len(X_train), len(y_train)))
print('(test) X: {}, y: {}'.format(len(X_test), len(y_test)))

## Actividades

#### Paso 1 (5 puntos):

Transforme los datos de reviews (train y test) a numéricos, preservando la cantidad de tokens suficientes para poder poder usar modelo Xception vista en clase. Y escale los datos transformados en caso de ser necesario.

### Inconsistencia en la definición de sentimiento

El enunciado de la actividad en los comentarios del código menciona que el sentimiento se asigna a 0 (negativo) para ratings `{1, 2}` y 1 (positivo) para el resto. Sin embargo, el código proporcionado en la celda `hNexcycAmV5O` define el sentimiento de la siguiente manera:

```python
data['sentiment'] = data['rating'].apply(lambda x: 0 if x in [1, 2, 3] else 1)
```

Esto significa que:
*   **Clase 0 (Negativa):** Ratings 1, 2 y 3.
*   **Clase 1 (Positiva):** Ratings 4 y 5.

Para mantener la reproducibilidad con la base entregada, **se conservará la lógica del código proporcionado**, donde los ratings 1, 2 y 3 se consideran negativos y los ratings 4 y 5 positivos.

### Exploración de datos para Paso 1

Antes de transformar los datos de reviews, realizaremos una exploración mínima para entender sus características, lo que nos permitirá tomar decisiones informadas sobre la longitud de secuencia (`sequence_length`) y el tamaño del vocabulario (`max_tokens`).

In [2]:
import tensorflow as tf
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd # Added pandas import
import seaborn as sns # Added seaborn import

# --- Configuración de semillas para reproducibilidad ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU') != []}")

# --- 1. Cantidad de observaciones ---
print(f"\nCantidad de observaciones en X_train: {len(X_train)}")
print(f"Cantidad de observaciones en X_test: {len(X_test)}")

# --- 2. Valores nulos ---
# X_train y X_test son listas de strings, no pandas Series/DataFrames.
# Los valores 'nan' se convirtieron a 'str(nan)' durante la conversión a lista.
# Vamos a verificar si hay 'nan' como string en las listas.
null_train = sum(1 for x in X_train if x == 'nan')
null_test = sum(1 for x in X_test if x == 'nan')
print(f"\nValores 'nan' (como string) en X_train: {null_train}")
print(f"Valores 'nan' (como string) en X_test: {null_test}")

# --- 3. Distribución de clases en y_train ---
class_distribution = pd.Series(y_train).value_counts(normalize=True)
print("\nDistribución de clases en y_train:")
print(class_distribution)

# --- 4. Longitud de las reviews en tokens ---
# Usaremos una tokenización simple por espacios para estimar las longitudes.
review_lengths = [len(review.split()) for review in X_train]

print("\nEstadísticas de longitud de reviews en X_train (en palabras):")
print(f"Min: {np.min(review_lengths)}")
print(f"Max: {np.max(review_lengths)}")
print(f"Mean: {np.mean(review_lengths):.2f}")
print(f"Median (P50): {np.percentile(review_lengths, 50)}")
print(f"P75: {np.percentile(review_lengths, 75)}")
print(f"P90: {np.percentile(review_lengths, 90)}")
print(f"P95: {np.percentile(review_lengths, 95)}")
print(f"P99: {np.percentile(review_lengths, 99)}")

# Visualización de la distribución de longitudes
plt.figure(figsize=(10, 6))
sns.histplot(review_lengths, bins=50, kde=True)
plt.title('Distribución de Longitud de Reviews en X_train')
plt.xlabel('Número de Palabras')
plt.ylabel('Frecuencia')
plt.grid(axis='y', alpha=0.75)
plt.show()

TensorFlow Version: 2.20.0
GPU Available: True


NameError: name 'X_train' is not defined

### Justificación de `sequence_length`

Basado en el análisis de la longitud de las reviews en `X_train` (medido en palabras):

*   **P50 (mediana):** 62 palabras
*   **P75:** 104 palabras
*   **P90:** 166 palabras
*   **P95:** 235 palabras
*   **P99:** 461 palabras

Para cubrir la gran mayoría de las reviews sin introducir un costo computacional excesivo debido a un padding extremo, seleccionaremos una `sequence_length` de **256**. Esta longitud cubre aproximadamente el 95% de las reviews, asegurando que la mayor parte de la información contextual se conserve. Reviews más largas serán truncadas y reviews más cortas serán rellenadas (padded) con el token `0`.

### Justificación de `max_tokens`

El tamaño del vocabulario (`max_tokens`) impacta directamente en la complejidad del modelo y la capacidad de representar palabras raras. Para esta tarea, utilizaremos un `max_tokens` de **20000**. Este valor es un compromiso razonable para capturar una gran parte del vocabulario más frecuente, que es el que generalmente aporta más información para la clasificación de sentimiento, mientras se mantiene el modelo manejable y se evitan palabras muy raras que podrían ser ruido o aumentar innecesariamente la dimensionalidad del embedding. El ajuste de vocabulario se realizará exclusivamente sobre `X_train` para evitar *data leakage*.

In [3]:
from tensorflow.keras.layers import TextVectorization

# Parámetros definidos
MAX_TOKENS = 20000 # Tamaño del vocabulario
SEQUENCE_LENGTH = 256 # Longitud de la secuencia de tokens

# Inicializar la capa TextVectorization
# `output_mode='int'` para obtener secuencias de enteros (índices de tokens)
# `output_sequence_length` para asegurar que todas las secuencias tengan la misma longitud
vectorize_layer = TextVectorization(
    max_tokens=MAX_TOKENS,
    output_mode='int',
    output_sequence_length=SEQUENCE_LENGTH,
    # Por defecto, el token 0 es para padding y el 1 para OOV (out-of-vocabulary).
    # El enunciado solicita usar 0 para padding. Mantendremos el comportamiento por defecto de TV.
)

# Adaptar la capa TextVectorization SOLAMENTE con los datos de entrenamiento
# Esto evita data leakage desde el conjunto de prueba
print("Adaptando TextVectorization a X_train...")
vectorize_layer.adapt(X_train)
print("Adaptación completada.")

# Obtener el vocabulario construido
vocabulary = vectorize_layer.get_vocabulary()
print(f"\nTamaño del vocabulario adaptado: {len(vocabulary)}")
# print(f"Ejemplo de vocabulario (primeras 20 palabras): {vocabulary[:20]}")

# Transformar los conjuntos de entrenamiento y prueba
X_train_tokenized = vectorize_layer(np.array(X_train))
X_test_tokenized = vectorize_layer(np.array(X_test))

# Convertir y_train e y_test a arrays de numpy si no lo son ya
y_train_np = np.array(y_train)
y_test_np = np.array(y_test)

print("\nTransformación de datos a secuencias numéricas completada.")

Adaptando TextVectorization a X_train...


NameError: name 'X_train' is not defined

### Evaluación de Escalamiento

Los datos transformados (`X_train_tokenized`, `X_test_tokenized`) son secuencias de índices enteros que representan tokens del vocabulario. Estos índices no tienen un significado numérico de magnitud, sino que actúan como identificadores categóricos para las palabras. Cuando estos índices se utilizan como entrada a una capa `Embedding` en un modelo de Deep Learning, la capa `Embedding` los convierte en vectores densos de punto flotante aprendibles.

Escalar estos índices enteros con un método como `StandardScaler` (que normaliza los valores restando la media y dividiendo por la desviación estándar) **destruiría su significado categórico** al convertirlos en valores flotantes sin relación con su identidad original en el vocabulario.

Por lo tanto, **no se necesita escalamiento** para los datos de entrada tokenizados cuando se usan con una capa `Embedding`.

### Resumen de `Paso 1`: Procesamiento de Texto

In [4]:
# Shapes de los conjuntos transformados
print(f"Shape de X_train_tokenized: {X_train_tokenized.shape}")
print(f"Shape de y_train: {y_train_np.shape}")
print(f"Shape de X_test_tokenized: {X_test_tokenized.shape}")
print(f"Shape de y_test: {y_test_np.shape}")

# Vocabulario utilizado
print(f"\nTamaño del vocabulario: {len(vocabulary)}")
print(f"Primeras 10 palabras del vocabulario: {vocabulary[:10]}")

# Longitud de secuencia
print(f"\nLongitud de secuencia utilizada: {SEQUENCE_LENGTH}")

# Ejemplo de review original y su representación tokenizada
example_index = 50 # Un índice arbitrario para mostrar un ejemplo
original_review = X_train[example_index]
tokenized_review = X_train_tokenized[example_index].numpy()

print(f"\nEjemplo de review original (índice {example_index}):\n{original_review}")
print(f"\nRepresentación tokenizada (primeros 20 tokens):\n{tokenized_review[:20]}...")
print(f"Representación tokenizada (últimos 20 tokens):\n...{tokenized_review[-20:]}")

# Distribución de clases en y_train (ya calculada, se puede volver a mostrar si es necesario)
print("\nDistribución de clases en y_train (clase 0: negativa, clase 1: positiva):")
print(pd.Series(y_train_np).value_counts(normalize=True))

# Distribución de clases en y_test
print("\nDistribución de clases en y_test (clase 0: negativa, clase 1: positiva):")
print(pd.Series(y_test_np).value_counts(normalize=True))

NameError: name 'X_train_tokenized' is not defined

#### Paso 2 (5 puntos):

Entrene el modelo Xception -- vista en clase -- con los datos procesados en el Paso 1. Entregue la matriz de confusión y reporte de clasificación con el conjunto de test.

#### Paso 3 (5 puntos):

Entrene un modelo Xception modificado (reemplazando solo la 2da capa Convolucional tradicional del modulo de "Entry Flow" por un bloque de Inception) con los datos procesados en el Paso 1. Entregue la matriz de confusión y reporte de clasificación con el conjunto de test.

#### Paso 4 (3 puntos):

Compare los resultados obtenidos en los Pasos 2 y 3 ¿Con cuál de los dos modelos se quedaría?. Justique su respuesta.